# Databricks Data Engineering Bootcamp
## Peripheral Module A: Mastering Slowly Changing Dimensions (SCD Type 0, 1, 2, 3)

**Context:** Managing historical mutations in master registries is a core responsibility of a Data Engineer. Depending on downstream analytical requirements, you must implement the correct SCD pattern:
1. **SCD Type 0 (Retain Original):** Attributes are immutable. Once written, incoming changes are completely ignored (e.g., Enrollment Dates).
2. **SCD Type 1 (Overwrite):** Updates values in place. No history is kept.
3. **SCD Type 3 (Add New Column):** Tracks limited history by storing the "Previous" and "Current" states side-by-side in the same row.
4. **SCD Type 2 (Add New Row):** Full history tracking via active status flags and validity date windows.

### 🔑 Architectural Alignment (Silver Layer Grain Constraint)
In compliance with enterprise data platform standards, all Slowly Changing Dimension transformations are executed strictly inside the Silver Layer. The Fact table grain remains completely protected because transactional tables will join against these standardized dimensional states.

**Task:** You will deploy all four SCD engines using PySpark and Delta Lake to analyze how an incoming change affects your physical tables.

### Chapter 1: Pipeline Baseline Setup
We simulate our base target state (Day 1) and the incoming daily delta updates (Day 2).

In [0]:
# silver_fact_sales vs silver_fact_sales_new
# se kathe grammi ftiaxneis to hash key kai to vazw se hash function. meta allo hask key poy einai to full row record hash key. kathe row einai anaparastasi tou hask key kai full row hask key. the next day an erthei id tha hask keys that parameinoun idia. an erthei idio id kai allo city to ena hask key tha einai idio alla to full row hask key tha einai diaforetiko

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType

# Day 2 Daily Staging Batch (Customer 101 moved to Patras! Customer 103 is brand new!)
delta_schema = StructType([
    StructField("customer_id", LongType(), True),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True)
])
incoming_df = spark.createDataFrame([(101, "Giannis", "Patras"), (103, "Eleni", "Thessaloniki")], delta_schema)

print("Incoming Delta Staging Batch (Day 2):")
display(incoming_df)

## Chapter 2: Strategy A — SCD Type 0: Retain Original (Static Master Data)

### SCD Type 0 Mechanics: Immutable Records
In SCD Type 0, attributes are considered completely static. Once a record is inserted for a specific customer_id, any downstream mutations or updates in the source staging system are explicitly ignored. 

![SCD Type 0](scd_type0.png)

In [0]:
from delta.tables import DeltaTable

# Initialize Target Table for Type 0
spark.sql("""
CREATE TABLE IF NOT EXISTS silver_cust_type0 (
    customer_id LONG,
    customer_name STRING,
    city STRING
) USING DELTA
""")

# --- SCD TYPE 0: INSERT-ONLY MERGE ---
# TODO: Use the correct operational match keyword to ensure Type 0 never alters existing records
DeltaTable.forName(spark, "silver_cust_type0").alias("target") \
    .merge(incoming_df.alias("source"), "target.customer_id = source.customer_id") \
    .___Insert(values = {
        "customer_id": "source.customer_id", 
        "customer_name": "source.customer_name", 
        "city": "source.city"
    }).execute()

print("SCD Type 0 Operational Execution Complete.")
display(spark.table("silver_cust_type0"))

## Chapter 3: Strategy B — SCD Type 1: Overwrite (No History Tracking)

### SCD Type 1 Mechanics: Inline Overwrite
SCD Type 1 updates historical attribute states in place. When a change arrives, the old value is permanently deleted and overwritten with the active state. No history is retained.

![SCD Type 1](scd_type1.png)

In [0]:
# Initialize Target Table for Type 1
spark.sql("""
CREATE TABLE IF NOT EXISTS silver_cust_type1 (
    customer_id LONG,
    customer_name STRING,
    city STRING
) USING DELTA
""")

# --- SCD TYPE 1: OVERWRITE MERGE ---
# TODO: Complete the Type 1 condition block to overwrite attributes on matching business keys
DeltaTable.forName(spark, "silver_cust_type1").alias("target") \
    .merge(incoming_df.alias("source"), "target.customer_id = source.customer_id") \
    .whenMatchedUpdate(set = {
        "customer_name": "source.customer_name",
        "___": "source.city" 
    }) \
    .whenNotMatchedInsert(values = {
        "customer_id": "source.customer_id", 
        "customer_name": "source.customer_name", 
        "city": "source.city"
    }).execute()

print("SCD Type 1 Operational Execution Complete.")
display(spark.table("silver_cust_type1"))

## Chapter 4: Strategy C — SCD Type 3 (Current vs Previous Column Tracking)

### SCD Type 3 Mechanics: Column-Level Dual Horizon Tracking
SCD Type 3 tracks a limited historical horizon by shifting values into dedicated parallel columns inside the same row. This keeps the schema light while allowing analysts to compare the "Current" vs "Previous" state directly.

![SCD Type 3](scd_type3.png)

In [0]:
# Initialize Target Table for Type 3
spark.sql("""
CREATE TABLE IF NOT EXISTS silver_cust_type3 (
    customer_id LONG, 
    customer_name STRING, 
    current_city STRING, 
    previous_city STRING
) USING DELTA
""")

# TODO: Complete the value mapping to shift the target's active city down into the previous tracking field
DeltaTable.forName(spark, "silver_cust_type3").alias("target") \
    .merge(incoming_df.alias("source"), "target.customer_id = source.customer_id") \
    .whenMatchedUpdate(
        condition = "target.current_city <> source.city",
        set = {
            "previous_city": "___", 
            "current_city": "source.city"            
        }
    ) \
    .whenNotMatchedInsert(values = {
        "customer_id": "source.customer_id",
        "customer_name": "source.customer_name",
        "current_city": "source.city",
        "previous_city": "null" 
    }).execute()

print("SCD Type 3 Operational Execution Complete.")
display(spark.table("silver_cust_type3"))

## Chapter 5: Strategy D — SCD Type 2 (Full Row Historization)

### SCD Type 2 Mechanics: Full Multi-Row Auditing & Versioning
SCD Type 2 is the gold standard for enterprise dimensional historization. Every single change prompts a split: the active record is closed out (is_current = False, end_date = current_timestamp), and a brand new record row is appended to house the active state.

![SCD Type 2](scd_type2.png)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# 1. Initialize Target Table for Type 2
spark.sql("""
CREATE TABLE IF NOT EXISTS silver_cust_type2 (
    customer_id LONG,
    customer_name STRING,
    city STRING,
    is_current BOOLEAN,
    start_date TIMESTAMP,
    end_date TIMESTAMP
) USING DELTA
""")

# 2. Format incoming update batch parameters with system tracking tags
updates_with_meta = incoming_df.withColumn("is_current", F.lit(True)) \
    .withColumn("start_date", F.current_timestamp()) \
    .withColumn("end_date", F.lit(None).cast("timestamp"))

# 3. Generate the historical rows required to split changing records
target_df = spark.table("silver_cust_type2").filter("is_current = true")

staged_updates = updates_with_meta.join(
    target_df, "customer_id", "inner"
).select(
    F.lit(None).cast("long").alias("customer_id"),
    updates_with_meta["customer_name"],
    updates_with_meta["city"],
    F.lit(False).alias("is_current"),
    target_df["start_date"],
    F.current_timestamp().alias("end_date")
).union(updates_with_meta)

# 4. Execute Type 2 Merge Transaction cleanly without syntax mismatches
DeltaTable.forName(spark, "silver_cust_type2").alias("target") \
    .merge(staged_updates.alias("source"), "target.customer_id = source.customer_id") \
    .whenMatchedUpdate(
        condition = "target.is_current = true AND target.city <> source.city",
        set = {
            "___": "false",
            "end_date": "source.end_date"
        }
    ) \
    .whenNotMatchedInsert(values = {
        "customer_id": "source.customer_id",
        "customer_name": "source.customer_name",
        "city": "source.city",
        "is_current": "source.is_current",
        "start_date": "source.start_date",
        "end_date": "source.end_date"
    }) \
    .execute()

# 5. Output and display the target status verification query
print("SCD Matrix Framework Operational Evaluation Complete.")
display(spark.table("silver_cust_type2").orderBy("customer_id", "start_date"))